In [ ]:
# Install required packages for JupyterLite/Pyodide
import piplite
await piplite.install(['scipy', 'ipywidgets', 'matplotlib'])
%matplotlib inline

# Why does Rosalind Franklin's X-ray diffraction image of DNA look the way it does?

In this notebook, we will explore the diffraction pattern of DNA, and see how it is related to the structure of the molecule. We will also see how the diffraction pattern can be used to determine the structure of the molecule.

# Exploring 1D diffraction patterns

We will look at the diffraction pattern of a simple 1D crystal, which is a repeating pattern of atoms in a line. We will see how the diffraction pattern changes as we change the spacing between the atoms.

In [ ]:
# import libraries
# numpy, ffts, matplotlib
import numpy as np
import matplotlib.pyplot as plt
from scipy.fftpack import fft, ifft, fftfreq, fftshift
import matplotlib.pyplot as plt
from ipywidgets import interact

In [ ]:
# function to convenienlty get the 1D and 2D FFTs
# they should return the frequency and the FFT of the signal

def get_1D_FFT(signal, dt=1.0):
    # get the length of the signal
    N = len(signal)
    # get the FFT of the signal
    signal_fft = fft(signal)
    # get the frequency
    freq = fftfreq(N, d=dt)
    return freq, signal_fft

def get_2D_FFT(signal, dt=1.0):
    # get the shape of the signal
    N, M = signal.shape
    # get the 2D FFT of the signal
    signal_fft = np.fft.fft2(signal)
    # get the frequency
    freq_x = fftfreq(N, d=dt)
    freq_y = fftfreq(M, d=dt)
    return freq_x, freq_y, signal_fft

# Make plots of the 1D and 2D FFTs of the following signals
def plot_1D_FFT(fft, fftfreq, fold = False):
    # plot the 1D FFT
    plt.figure()
    if fold:
        plt.plot(fftfreq[:fftfreq.size//2], np.abs(fft)[:fftfreq.size//2])
    else:
        plt.plot(fftshift(fftfreq), fftshift(np.abs(fft)))
    plt.xlabel('Frequency')
    plt.ylabel('Amplitude')
    plt.show()
    
def plot_2D_FFT(fft, freq_x, freq_y, show_axes = False):
    # plot the 2D FFT
    plt.figure()
    plt.imshow(np.abs(fft), extent=(freq_x.min(), freq_x.max(), freq_y.min(), freq_y.max()))
    plt.xlabel('Frequency x')
    plt.ylabel('Frequency y')
    plt.colorbar()
    
    # add external axes to show the frequency
    if show_axes:
        plt.twinx().plot(freq_x, np.zeros_like(freq_x), 'k')
        plt.twiny().plot(np.zeros_like(freq_y), freq_y, 'k')
    
    
    
    plt.show()

In [ ]:
# of a sin wave
def generate_sin(freq, signal_length, phase = 0, fs = 100):
    # time step
    dt = 1/fs
    # time array
    t = np.arange(0, signal_length, dt)
    signal = np.sin(2*np.pi*freq*t + phase)
    return t, signal


In [ ]:
# create a function for displaying a signal and its fft
# the plots should but one on top of the other, with a gap between them


def plot_signal_and_fft(signal, t, fold = False):
    fig, axs = plt.subplots(2, 1, figsize=(10, 6))
    axs[0].plot(t, signal)
    axs[0].set_xlabel('Time')
    axs[0].set_ylabel('Amplitude')
    freq, fft = get_1D_FFT(signal, t[1]-t[0])
    if not fold:
        axs[1].plot(fftshift(freq), np.abs(fftshift(fft)))
    else:
        axs[1].plot(freq[:freq.size//2], np.abs(fft)[:freq.size//2])
    axs[1].set_xlabel('Frequency')
    axs[1].set_ylabel('Amplitude')
    plt.show()
    
# 2d 
def plot_image_and_fft(image, xrange = None, yrange = None, logfft = False):
    fig, axs = plt.subplots(1,2, figsize=(10, 6))
    xrange = xrange or [0, image.shape[0]]
    yrange = yrange or [0, image.shape[1]]
    axs[0].imshow(image, cmap='gray', extent=(xrange[0], xrange[1], yrange[0], yrange[1]))
    axs[0].set_xlabel('x')
    axs[0].set_ylabel('y')
    freq_x, freq_y, fft = get_2D_FFT(image)
    # shift so 0 is in the center
    fft = fftshift(fft)
    freq_x = fftshift(freq_x)
    freq_y = fftshift(freq_y)
    # iff logfft is True, plot the log of the fft
    im = axs[1].imshow(np.abs(fft), extent=(freq_x.min(), freq_x.max(), freq_y.min(), freq_y.max()))
    axs[1].set_xlabel('Frequency x')
    axs[1].set_ylabel('Frequency y')
    # properly sized, narrow colorbar
    # close to the image
    cbar = plt.colorbar(im, ax=axs[1], fraction=0.046, pad=0.04)

    
    plt.show()

In [ ]:
def double_slit(sep, height = 20, width = 3, size = 100):
    image = np.zeros((size, size))
    image[size//2-height//2:size//2+height//2, size//2-sep-width//2:size//2-sep+width//2] = 1
    image[size//2-height//2:size//2+height//2, size//2+sep-width//2:size//2+sep+width//2] = 1
    return image

In [ ]:
# Time to do dna

# lets create a helix
# we will use the parametric equations for a helix
# x = r*cos(t)
# y = r*sin(t)
# z = at
# where r is the radius of the helix
# and a is the pitch of the helix
def helix_3d(r, a, t):
    x = r*np.cos(t)
    y = r*np.sin(t)
    z = a*t
    return x, y, z

def helix_2d_math(r, a, t, phi = 0):
    y = r*np.sin(t + phi)
    z = a*t
    return y, z


# version of the 2d helix, where z = t instead of at, so the period must change
def helix_2d(r, a, t, phi = 0, delta = 0):
    # assume a is the pitch in Angstrom
    # that t is in Angstroms
    # and that r is in Angstroms
    y = r*np.sin(2*np.pi*t/a + phi)
    z = t - delta
    return y, z


In [ ]:


def plot_dna(pitch):
    image_size = 300
    t = np.linspace(0, 200, image_size**2)
    y, z = helix_2d(r, pitch, t)
    scale = image_size / z.max()
    y = y * scale
    z = z * scale
    image = np.zeros((image_size, image_size))
    image[z.astype(int) -1, y.astype(int)-1] = 1
    image = fftshift(image, axes=1)
    plot_image_and_fft(image)
    


In [ ]:
# interactive double helix for dna changing phase and pitch
def plot_dna_double(pitch, phase):
    image_size = 300
    t = np.linspace(0, 200, image_size**2)
    y, z = helix_2d(r, pitch, t)
    scale = image_size / z.max()
    y = y * scale
    z = z * scale
    image = np.zeros((image_size, image_size))
    image[z.astype(int) -1, y.astype(int)-1] = 1
    
    # add second helix
    # this is dna, so just shift by 23 degrees phase
    y, z = helix_2d(r, pitch, t, phi = phase * np.pi/180)
    y = y * scale
    z = z * scale
    image[z.astype(int) -1, y.astype(int)-1] = 1
    image = fftshift(image, axes=1)
    plot_image_and_fft(image)
    


In [ ]:
# interactive double helix for dna changing phase and pitch
def plot_dna_double_bp(pitch = 36, phase = 133, bp_per_turn = 10, show_helix = True, show_single = True, show_double = True, show_bp = True):
    image_size = 500
    
    r = 18 # angstrom
    a = 33.2 #angstrom
    t = np.linspace(0, 200, image_size**2)
    y, z = helix_2d(r, pitch, t)
    scale = image_size / z.max()
    y = y * scale
    z = z * scale
    image = np.zeros((image_size, image_size))
    if show_single and show_helix:
        image[z.astype(int) -1, y.astype(int)-1] = 1
    
    # add second helix
    # this is dna, so just shift by 23 degrees phase
    y2, z2 = helix_2d(r, pitch, t, phi = phase * np.pi/180)
    y2 = y2 * scale
    z2 = z2 * scale
    if show_helix and show_double:
        image[z2.astype(int) -1, y2.astype(int)-1] = 1
    
    image = fftshift(image, axes=1)
    
    if show_bp:
        # add base pairs every 10 angstroms
        t = np.arange(0, 200, pitch / bp_per_turn)
        y, z = helix_2d(r, pitch, t)
        y2, z2 = helix_2d(r, pitch, t, phi = phase * np.pi/180)
        y = y * scale  + image_size // 2
        z = z * scale
        y2 = y2 * scale + image_size // 2
        z2 = z2 * scale
            
        for yy in zip(z, y, y2):
            # print(yy[0].astype(int), min(yy[1],yy[2]).astype(int), max(yy[1],yy[2]).astype(int))
            # plt.plot([yy[1], yy[2]], [yy[0], yy[0]], 'k')
            image[yy[0].astype(int), min(yy[1],yy[2]).astype(int):max(yy[1],yy[2]).astype(int)-1] = 1
            
    # add base pairs
    
    
    
    
    fig, axs = plt.subplots(1,2, figsize=(10, 6))
    xrange = None or [0, image.shape[0]]
    yrange = None or [0, image.shape[1]]
    axs[0].imshow(image, cmap='gray', extent=(xrange[0], xrange[1], yrange[0], yrange[1]))
    axs[0].set_xlabel('x')
    axs[0].set_ylabel('y')
    axs[0].set_aspect('equal')
    freq_x, freq_y, fft = get_2D_FFT(image)
    # shift so 0 is in the center
    fft = fftshift(fft)
    freq_x = fftshift(freq_x)
    freq_y = fftshift(freq_y)
    # iff logfft is True, plot the log of the fft
    im = axs[1].imshow(np.sign(np.real(fft)) * np.abs(fft), extent=(freq_x.min(), freq_x.max(), freq_y.min(), freq_y.max()), vmin=-1000, vmax = 1000, cmap='RdBu')
    axs[1].set_xlabel('Frequency x')
    axs[1].set_ylabel('Frequency y')
    # zoom in on the center
    axs[1].set_xlim(-0.15, 0.15)
    axs[1].set_ylim(-0.15, 0.15)
    axs[1].set_aspect('equal')
    # properly sized, narrow colorbar
    # close to the image
    cbar = plt.colorbar(im, ax=axs[1], fraction=0.046, pad=0.04)
    
interact(plot_dna_double_bp, pitch=(1, 100, 1), phase=(0, 360, 1), bp_per_turn=(1, 20, 1))

